## 1️⃣ Importation & Exploration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("bank.csv")
df.columns = df.columns.str.lower().str.strip()

print(df.dtypes)
print(df.shape)
df.info()
print(df.describe())

v_manque = df.isnull().sum() / df.shape[0] * 100
plt.bar(v_manque.index, v_manque.values)
plt.xticks(rotation=90)
for ind, vl in enumerate(v_manque.values):
    if vl > 0:
        plt.text(ind, vl, str(vl.round(2)), ha="center", va="bottom")
plt.show()

duplicate = df[df.duplicated(subset=["transaction_id"], keep=False)].copy()
duplicate["date_temp"] = pd.to_datetime(duplicate["date_transaction"], errors="coerce", dayfirst=True)
duplicate["ecart_jours"] = duplicate.groupby("transaction_id")["date_temp"].diff().dt.days
df.head()

## 2️⃣ Nettoyage des données

In [ ]:
df.drop_duplicates(subset="transaction_id", keep="first", inplace=True)

df["date_transaction"] = pd.to_datetime(df["date_transaction"], errors="coerce", dayfirst=True)

df["montant"] = df["montant"].astype(str).str.replace(",", ".")
df["montant"] = pd.to_numeric(df["montant"], errors="coerce")

df["solde_avant"] = df["solde_avant"].astype(str).str.replace(" EUR", "")
df["solde_avant"] = pd.to_numeric(df["solde_avant"], errors="coerce")

df["devise"] = df["devise"].str.upper().str.strip()
df["segment_client"] = df["segment_client"].str.capitalize()
df["agence"] = df["agence"].str.strip()

df["score_credit_client"] = df["score_credit_client"].fillna(df["score_credit_client"].median())
df["agence"] = df["agence"].fillna(df["agence"].mode()[0])
df["segment_client"] = df["segment_client"].fillna(df["segment_client"].mode()[0])
df["date_transaction"] = df["date_transaction"].fillna(df["date_transaction"].mode()[0])

df = df.drop(columns="taux_interet", errors="ignore")

## Détection des Valeurs Aberrantes

In [ ]:
Q1 = df["montant"].quantile(0.25)
Q3 = df["montant"].quantile(0.75)
IQR = Q3 - Q1
anomalie_montant = (df["montant"] < (Q1 - 1.5 * IQR)) | (df["montant"] > (Q3 + 1.5 * IQR))

Q1_1 = df["score_credit_client"].quantile(0.25)
Q3_3 = df["score_credit_client"].quantile(0.75)
IQR_score = Q3_3 - Q1_1
anomalie_score_credit = (df["score_credit_client"] < (Q1_1 - 1.5 * IQR_score)) | (df["score_credit_client"] > (Q3_3 + 1.5 * IQR_score))

anomalie_score = (df["score_credit_client"] < 0) | (df["score_credit_client"] > 850)

df["is_anomalie"] = anomalie_montant | anomalie_score_credit | anomalie_score

## Feature Engineering

In [ ]:
df["annee"] = df["date_transaction"].dt.year
df["mois"] = df["date_transaction"].dt.month
df["trimestre"] = df["date_transaction"].dt.quarter
df["semaine-jour"] = df["date_transaction"].dt.dayofweek

df["montant_eur_verifie"] = (df["montant"] / df["taux_change_eur"])
comparer = (df["montant_eur"]).compare(df["montant_eur_verifie"])

def risque(ca):
    if ca >= 700: return "Low"
    elif ca >= 580: return "Medium"
    else: return "High"

df["categorie_risque"] = df["score_credit_client"].apply(risque)

total_credit = df[df["type_operation"] == "Credit"].groupby("client_id")["montant"].sum()
total_debit = df[df["type_operation"] == "Debit"].groupby("client_id")["montant"].sum()

solde = pd.DataFrame({"total_credit": total_credit, "total_debit": total_debit}).fillna(0).reset_index()
df = pd.merge(df, solde, on="client_id", how="left")
df["solde_net"] = (df["total_credit"] - df["total_debit"])

stats = df.groupby("client_id").agg(
    nb_transaction=("transaction_id", "count"),
    montant_moyen=("montant", "mean"),
    nb_produit=("produit", "nunique")
).reset_index()

df = pd.merge(df, stats, on="client_id", how="left")

df["taux_rejet"] = df.groupby('agence')['statut'].transform(lambda x: (x == 'Rejete').mean() * 100)

## Export

In [ ]:
df.to_csv("financecore_clean.csv", index=False)

decisions = """# DECISIONS.md - FinanceCore SA
- Doublons : Supprimés sur transaction_id en gardant la première occurrence.
- Dates : Unifiées au format AAAA-MM-JJ HH:MM:SS.
- Montants : Séparateur corrigé (virgule vers point) et conversion en float.
- Solde : Nettoyage du texte ' EUR' et conversion.
- Textes : Devises en majuscules, segment unifié, espaces supprimés.
- Valeurs manquantes : Imputation par la médiane (score crédit) et le mode (agence, segment).
- Anomalies : Détectées par IQR et limites métier (0-850), marquées avec is_anomalie.
- Feature Engineering : Création des indicateurs (solde net, taux de rejet, catégories de risque).
"""
with open("DECISIONS.md", "w", encoding="utf-8") as f:
    f.write(decisions)